# Training data and predictor improvement workflow

In [1]:
import ee

ee.Authenticate()
ee.Initialize()

In [2]:
!python -m pip install .. --quiet

# Define AOI

In [3]:
import geemap

# Choose AOI

aoi = geemap.shp_to_ee('../data/area_of_interest.shp')

# Satellite data

In [4]:
from luma_ge.data_acquisition import Reflectance_Data, final_Image

optical_reflectance = Reflectance_Data()

composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2018-01-01'
end = '2018-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L8_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True,
                                                            coverage_scale=100)

2026-07-13 16:59:34,766 - luma_ge.ee_config - INFO - Earth Engine initialized successfully
2026-07-13 16:59:34,767 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-07-13 16:59:34,768 - final_Image - INFO - final_Image creation initialized.
2026-07-13 16:59:34,769 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-07-13 16:59:34,771 - Reflectance_Data - INFO - Date range: 2018-01-01 to 2018-12-31
2026-07-13 16:59:34,773 - Reflectance_Data - INFO - Cloud cover threshold: 40%
2026-07-13 16:59:34,773 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-07-13 16:59:34,775 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-07-13 16:59:34,776 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-07-13 16:59:35,078 - final_Image - INFO - Creating quality mosaic from 5 images using NDVI as quality metric
2026-07-13 16:59:35,080

# Classification scheme

In [5]:
import pandas as pd
from luma_ge.classification_scheme import LULC_Scheme_Manager

#Reset manager for CSV upload example
manager = LULC_Scheme_Manager()
#path to csv 
csv_path = "../data/Tes_classification_scheme.csv"

# Load the CSV
df = pd.read_csv(csv_path, sep=None, engine="python")
id_col, name_col, color_col = manager.auto_detect_csv_columns(df)
success, message = manager.process_csv_upload(df, id_col, name_col, color_col)
if success:
    print(f"✅ {message}")
    
    # Finalize the upload
    success, message = manager.finalize_csv_upload()
    if success:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print(f"❌ {message}")
    
classification_df = manager.get_dataframe()

✅ Successfully loaded 4 classes from CSV with colors from CSV
✅ Skema klasifikasi berhasil dibuat dengan 4 kelas


# Training data

In [6]:
TrainVectPath  = '../data/training_points.shp'
TrainField = 'ID' 

from luma_ge.sample_data import SyncTrainData

TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=classification_df,
            aoi_geometry=aoi,
            training_shp_path=TrainVectPath
        )

# Set class field
TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)

# Validate classes
TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, use_class_ids=True)

    # Check sample sufficiency
TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)

    # Filter by AOI
TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)

TrainDataFinal = TrainDataDict.get('training_data')

# Create summary table
table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
            training_data=TrainDataDict.get('training_data'),
            landcover_df=TrainDataDict.get('landcover_df'),
            class_field=TrainDataDict.get('class_field')
        )

display(table_df)

2026-07-13 16:59:39,568 - luma_ge.sample_data - INFO - Loading training data from shapefile: ../data/training_points.shp
2026-07-13 16:59:39,662 - luma_ge.sample_data - WARNING - 'kelas' field not found in training data
2026-07-13 16:59:39,664 - luma_ge.sample_data - INFO - Available columns: ['LULC_Type', 'ID', 'geometry']
2026-07-13 16:59:39,665 - luma_ge.sample_data - INFO - Validating classes with use_class_ids=True
2026-07-13 16:59:39,667 - luma_ge.sample_data - INFO - Class field: ID
2026-07-13 16:59:39,668 - luma_ge.sample_data - INFO - Training data type: <class 'geopandas.geodataframe.GeoDataFrame'>
2026-07-13 16:59:39,669 - luma_ge.sample_data - INFO - Landcover DF columns: ['ID', 'Land Cover Class', 'Color Palette']
2026-07-13 16:59:39,671 - luma_ge.sample_data - INFO - Valid IDs in landcover_df: [1, 2, 3, 4]
2026-07-13 16:59:39,673 - luma_ge.sample_data - INFO - Processing GeoDataFrame with 44 features
2026-07-13 16:59:39,674 - luma_ge.sample_data - INFO - Unique classes in

,ID,LULC_class,Sample_Count,Percentage
0,3,3,12,27.272727
1,4,4,12,27.272727
2,2,2,10,22.727273
3,1,1,10,22.727273


# Sample data quality

In [7]:
from luma_ge.sample_data_quality import sample_quality

labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

#Conduct the analysis
analyzer = sample_quality(training_data=labeled_roi, 
    image= median_landsat, 
    class_property='ID',           # Column with numeric IDs (1, 2, 3, etc.)
    region= aoi,
    class_name_property='LULC_class'          # Column with names ('Forest', 'Urban', 'Water', etc.)
)

sample_df = analyzer.get_sample_stats_df()
display(sample_df)

# Extract spectral values
pixel_extract = analyzer.extract_spectral_values(scale=30, max_pixels_per_class=5000)

# Perform separability Analysis
separability_analysis = analyzer.get_separability_df(pixel_extract, method='TD')
lowest_sep = analyzer.lowest_separability(pixel_extract)
display(lowest_sep)

2026-07-13 16:59:39,724 - pyogrio._io - INFO - Created 44 records


,ID,Sample_Count,Proportion,Percentage
0,1,10,0.2273,22.73
1,2,10,0.2273,22.73
2,3,12,0.2727,27.27
3,4,12,0.2727,27.27


Extracted spectral values for 44 samples across 4 classes


,Class1_ID,Class1_Name,Class2_ID,Class2_Name,TD_Distance,Separability_Level,Interpretation
0,1,Class 1,2,Class 2,1.957,🔴 Poor Separability (TD < 1.999),🔴 Poor Separability (TD < 1.999)
1,2,Class 2,4,Class 4,1.978,🔴 Poor Separability (TD < 1.999),🔴 Poor Separability (TD < 1.999)
2,1,Class 1,4,Class 4,2.000,🟢 Good Separability (TD ≥ 1.999),🟢 Good Separability (TD ≥ 1.999)
3,1,Class 1,3,Class 3,2.000,🟢 Good Separability (TD ≥ 1.999),🟢 Good Separability (TD ≥ 1.999)
4,2,Class 2,3,Class 3,2.000,🟢 Good Separability (TD ≥ 1.999),🟢 Good Separability (TD ≥ 1.999)
5,4,Class 4,3,Class 3,2.000,🟢 Good Separability (TD ≥ 1.999),🟢 Good Separability (TD ≥ 1.999)


# Predictor selection based on separability analysis result



In [8]:
# Take the worst pair from the existing separability output
worst = lowest_sep.iloc[0]
bands_df, overlap_df = analyzer.diagnose_pair(
    pixel_extract, worst['Class1_ID'], worst['Class2_ID']
)
display(bands_df)   # bands ranked by how little they separate this pair
display(overlap_df) # actual training points likely causing the confusion


Weakest bands, 1 vs 2:
   Band  TD_Distance
    NIR        0.038
AEROSOL        0.081
   BLUE        0.175
  SWIR2        0.219
    RED        0.285
0 overlapping/candidate-mislabeled samples flagged.


,Band,TD_Distance
0,NIR,0.038
1,AEROSOL,0.081
2,BLUE,0.175
3,SWIR2,0.219
4,RED,0.285
5,SWIR1,0.381
6,GREEN,0.443


,ID,dist_own,dist_other,AEROSOL,BLUE,GREEN,NIR,RED,SWIR1,SWIR2
